<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L03-five-workflow-patterns-with-evals/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L03-five-workflow-patterns-with-evals/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T08-L03-five-workflow-patterns-with-evals/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T08-L03-five-workflow-patterns-with-evals/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T08-L03 · Five workflow patterns in plain Python, each with an eval: chaining, routing, parallelisation, orchestration and evaluate-and-refine

**You will build:** the five composition patterns that sit between one prompt and a free-running agent, each as a
short function over a model, and the evaluation that says whether each one earned its extra calls.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, standard library only · **Prerequisites:** T07-L01-agent-loop-from-scratch, T07-L03-evaluating-agents

T07-L01 built the loop that lets a model decide its own next step. Before you let it, there is a cheaper question:
does the task need a loop at all? A **workflow** is a fixed path of model calls that your code controls. An
**agent** chooses its own path. This lesson builds the five standard workflows and measures each against a single call.

As of 20 October 2026, Anthropic's article "Building effective agents" draws the same line (workflows run through predefined code
paths; in agents the model directs its own process) and advises starting with simple prompts and adding multi-step systems only when
simpler solutions fall short. This lesson takes the second piece of advice as its method: a single-call baseline first, then the structure.

The model here is a **script**, not a language model, so that every tendency is written down, printed and
repeatable. What you measure is what those stated rules do. It is not a rate for any real model.

By the end you will be able to:

1. Implement prompt chaining with a programmatic gate and a retry on each step.
2. Implement routing that normalises a classifier's label and falls back when the label is unknown.
3. Implement parallel voting and a majority rule with a stated tie-break.
4. Implement an orchestrator that plans sub-tasks it did not know in advance, runs one worker per sub-task and synthesises.
5. Implement evaluate-and-refine with a round cap, and measure for every pattern its accuracy, its calls per task and the failure modes it adds, against a single call.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L01`, `T07-L03`

**This lesson assumes you can already:**

- write a Python function that loops over a list, builds strings with f-strings and returns a dict
- read a model call as ask(prompt) returning text, the way the agent loop of T07-L01 treats it, and count correct episodes as T07-L03 does
- use collections.Counter and fractions.Fraction, and read a table of counts of wins and losses on the same tasks

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) · Anthropic · *free* — where the five workflows (prompt chaining, routing, parallelization, orchestrator-workers, evaluator-optimizer) and the advice to start simple are written down; this lesson builds each one
- [Agent patterns (basic_workflows, orchestrator_workers, evaluator_optimizer)](https://github.com/anthropics/anthropic-cookbook/tree/main/patterns/agents) · Anthropic Cookbook · *free* — runnable notebooks of the same patterns against a real model, for the step this lesson leaves out (replace SimModel by a class whose ask calls the model)
- [collections: Counter](https://docs.python.org/3/library/collections.html#collections.Counter) · Python documentation · *free* — counting votes; Counter keeps first-seen order, which exercise 3's tie rule depends on
- [fractions: rational numbers](https://docs.python.org/3/library/fractions.html) · Python documentation · *free* — the exact agreement and rate values that exercise 3 and the sign test use

In [ ]:
# Setup: standard library only; versions printed.
import random
import re
import sys
from collections import Counter
from decimal import Decimal
from fractions import Fraction
from math import comb
from typing import Callable

print("python", sys.version.split()[0])

SEED = "T08-L03"           # every random draw below is seeded from this string plus a purpose, so a re-run is identical

_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, name shown, stubs, where).
_EXERCISES = [("exercise 1", "chain", ("chain",), "section 2"),
              ("exercise 2", "route", ("route",), "section 3"),
              ("exercise 3", "majority and parallel_vote", ("majority", "parallel_vote"), "section 4"),
              ("exercise 4", "parse_plan and orchestrate", ("parse_plan", "orchestrate"), "section 5"),
              ("exercise 5", "refine", ("refine",), "section 6")]
_SHOWN = {label: shown for label, shown, _, _ in _EXERCISES}
_STUBS = {label: stubs for label, _, stubs, _ in _EXERCISES}


def _unfinished(exc: BaseException) -> str:
    """The name of the function that raised NotImplementedError: the innermost frame."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (majority and parallel_vote)"; several -> "exercises 3 and 4"; a demo -> its own name."""
    if len(labels) > 1 and all(label in _SHOWN for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} ({_SHOWN[label]})" if label in _SHOWN else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 10     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own message,
    which names the likely mistake, and the notebook carries on, so one broken exercise never hides the
    feedback on the others. A demo names the exercises it `needs`: until each has passed its check, the
    demo says which one it is waiting for and skips. Every outcome is recorded for the progress board, and
    a script run exits non-zero at the end if any check came back wrong. A check that runs longer than
    its limit is stopped and counts as failed, so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, stubs in _STUBS.items() if stub in stubs and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _SHOWN:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The stand-in model, and what it is planted to get wrong

`SimModel.ask(prompt, sample=0)` answers a prompt of the form `skill: text`. A header such as `extract|total|format`
asks for several jobs in one call; each job reads the previous job's output. Four rules make the patterns worth
comparing. They are assumptions, written before any run, and the cell below prints every rate:

* **Load.** A call that packs in more separate jobs slips more often on each job (a fixed extra rate per added job).
* **Samples.** The same prompt with the same `sample` always gives the same answer. A different `sample` is a fresh draw,
  like asking again at a non-zero temperature. Repeating a call with the same `sample` therefore tells you nothing new.
* **Correlated errors.** On some labelling items the model is biased toward one fixed wrong label (the decoy), so repeated draws agree on the mistake.
* **Regressions.** A revision usually fixes the problem it is shown, sometimes does not, and sometimes breaks a different constraint.

The `scale` argument of `SimModel` multiplies every slip rate. At scale zero the model never slips, which will matter in section 8.
Each model object gets its own stream string, so every episode is repeatable alone and no result depends on run order.
You do not need to read the cell below to do the exercises; it is given. Read the printed rates.

In [ ]:
# --- given: the world, the planted rates and SimModel ---
PLANTED = {   # name -> slip probability before scaling: the author's assumptions
    "extra, per added job in one prompt": Fraction(7, 100),
    "extract": Fraction(6, 100), "total": Fraction(8, 100), "format": Fraction(5, 100),
    "classify (wrong label)": Fraction(4, 100),
    "refund": Fraction(5, 100), "shipping": Fraction(5, 100), "reset": Fraction(5, 100),
    "answer (one prompt covering all three kinds)": Fraction(5, 100),
    "label, easy item": Fraction(15, 100), "label, hard item (picks the decoy)": Fraction(60, 100),
    "plan drops a document": Fraction(6, 100), "plan repeats a document": Fraction(3, 100),
    "digest (one document)": Fraction(5, 100), "digest_all, extra per added document": Fraction(3, 100),
    "merge drops a line": Fraction(4, 100),
    "draft breaks each constraint": Fraction(25, 100),
    "revise leaves the problem": Fraction(15, 100), "revise breaks another constraint": Fraction(10, 100),
}
EXTRA = float(PLANTED["extra, per added job in one prompt"])
LOAD = {"answer": 3}                       # the one-prompt-for-everything handler carries three kinds of job
HARD_SHARE = Fraction(1, 4)                # share of labelling items that are hard
LABELS = ["bug", "billing", "praise"]
CATEGORIES = ["refund", "shipping", "reset"]
DAYS = {"DE": 2, "FR": 3, "US": 6, "JP": 9}
N_TASKS = 200                              # tasks per pattern, each from its own seed string
CONSTRAINTS = ["oid", "short", "tone", "signoff"]
VOTES, RETRIES, ROUNDS = 5, 1, 3           # the settings each pattern runs with in the tables


def P(name: str) -> float:
    return float(PLANTED[name])


# --- chain world: an order line -> record -> record with a total -> a sentence ---
NAMES = ["ana costa", "ben okafor", "chen li", "dara nolan", "eli moss", "faye ruiz"]
ITEMS_C = ["widget", "gadget", "sprocket", "bracket", "gasket"]
CURS = ["EUR", "USD", "GBP"]
RECORD = re.compile(r"[A-Z][a-z]*( [A-Z][a-z]*)*\|\d+\|[a-z]+\|\d+\.\d\d\|[A-Z]{3}")
SENTENCE = re.compile(r"[A-Z][a-z]*( [A-Z][a-z]*)* owes \d+\.\d\d [A-Z]{3} for \d+ x [a-z]+")


def _extract_true(raw: str) -> str:
    m = re.fullmatch(r"\s*([^;]+?)\s*;\s*(\d+)\s*x\s*([a-z]+)\s*;\s*([a-z]{3})\s+(\d+(?:\.\d+)?)\s*", raw, re.I)
    if not m:
        return "ERROR: unreadable order"
    name, qty, item, cur, price = m.groups()
    return f"{name.title()}|{int(qty)}|{item.lower()}|{Decimal(price):.2f}|{cur.upper()}"


def _extract_wrong(raw: str, rng: random.Random) -> str:
    parts = _extract_true(raw).split("|")
    if len(parts) != 5:
        return "ERROR: unreadable order"
    if rng.random() < 0.5:                 # a plausible wrong quantity: the shape stays valid
        parts[1] = str(int(parts[1]) + 1)
        return "|".join(parts)
    return "|".join(parts[:4])             # the currency is lost: the shape breaks


def _total_true(rec: str) -> str:
    parts = rec.split("|")
    if len(parts) != 5 or not parts[1].isdigit():
        return "ERROR: bad record"
    return rec + "|" + f"{Decimal(parts[1]) * Decimal(parts[3]):.2f}"


def _total_wrong(rec: str, rng: random.Random) -> str:
    parts = rec.split("|")
    if len(parts) != 5 or not parts[1].isdigit():
        return "ERROR: bad record"
    t = Decimal(parts[1]) * Decimal(parts[3])
    if rng.random() < 0.5:                 # one price too many: the arithmetic no longer holds
        return rec + "|" + f"{t + Decimal(parts[3]):.2f}"
    return rec + "|" + f"{t:.1f}"          # one decimal place: the shape breaks


def _format_true(rec6: str) -> str:
    p = rec6.split("|")
    if len(p) != 6:
        return "ERROR: bad record"
    return f"{p[0]} owes {p[5]} {p[4]} for {p[1]} x {p[2]}"


def _format_wrong(rec6: str, rng: random.Random) -> str:
    p = rec6.split("|")
    if len(p) != 6:
        return "ERROR: bad record"
    if rng.random() < 0.5:                 # the currency is dropped: the shape breaks
        return f"{p[0]} owes {p[5]} for {p[1]} x {p[2]}"
    return f"{p[0]} owes {p[5]} {p[4]} for {int(p[1]) + 1} x {p[2]}"     # a wrong count in a valid sentence


def gate_record(out: str) -> bool:
    """Programmatic check on step 1: five fields of the right shapes."""
    return bool(RECORD.fullmatch(out))


def gate_total(out: str) -> bool:
    """Programmatic check on step 2: the shapes hold AND quantity times price equals the total."""
    p = out.split("|")
    return (len(p) == 6 and bool(RECORD.fullmatch("|".join(p[:5]))) and bool(re.fullmatch(r"\d+\.\d\d", p[5]))
            and Decimal(p[1]) * Decimal(p[3]) == Decimal(p[5]))


def gate_sentence(out: str) -> bool:
    """Programmatic check on step 3: the sentence has the right shape. It cannot know the right count."""
    return bool(SENTENCE.fullmatch(out))


def chain_task(i: int) -> dict:
    rng = random.Random(f"{SEED}|task|chain|{i}")
    name, qty, item, cur = rng.choice(NAMES), rng.randint(2, 9), rng.choice(ITEMS_C), rng.choice(CURS)
    price = Decimal(rng.randint(150, 4999)) / 100

    def pad(s):
        return rng.choice(["", " ", "  "]) + s + rng.choice(["", " ", "  "])

    raw = f"{pad(rng.choice([name, name.upper(), name.title()]))};{pad(str(qty))}x {item};{pad(rng.choice([cur, cur.lower()]))} {price}"
    return {"raw": raw, "expected": f"{name.title()} owes {qty * price:.2f} {cur} for {qty} x {item}"}


# --- route world: three kinds of customer message, each with one exact reply ---
def _refund_msg(m):
    r = re.fullmatch(r"please refund (\d+) items at (\d+\.\d\d) each for order (\w+)", m)
    return r.groups() if r else None


def _ship_msg(m):
    r = re.fullmatch(r"when will order (\w+) arrive in (\w+)", m)
    return r.groups() if r else None


def _reset_msg(m):
    r = re.fullmatch(r"reset the password for (\w+)@example\.com", m)
    return r.groups() if r else None


def _refund(m, slip=False):
    g = _refund_msg(m)
    if g is None:
        return "ERROR: not a refund request"
    return f"REFUND {int(g[0]) * Decimal(g[1]) + (Decimal(g[1]) if slip else 0):.2f} {g[2]}"


def _shipping(m, slip=False):
    g = _ship_msg(m)
    if g is None:
        return "ERROR: not a shipping question"
    return f"SHIP {g[0]} {DAYS[g[1]] + (1 if slip else 0)} days"


def _reset(m, slip=False):
    g = _reset_msg(m)
    if g is None:
        return "ERROR: not a reset request"
    return f"RESET {g[0]}{'s' if slip else ''}@example.com"


HANDLERS = {"refund": _refund, "shipping": _shipping, "reset": _reset}


def category_of(m: str):
    for c, probe in (("refund", _refund_msg), ("shipping", _ship_msg), ("reset", _reset_msg)):
        if probe(m):
            return c
    return None


def route_task(i: int) -> dict:
    rng = random.Random(f"{SEED}|task|route|{i}")
    cat, oid = rng.choice(CATEGORIES), f"A{rng.randint(1000, 9999)}"
    if cat == "refund":
        msg = f"please refund {rng.randint(2, 6)} items at {Decimal(rng.randint(300, 4000)) / 100:.2f} each for order {oid}"
    elif cat == "shipping":
        msg = f"when will order {oid} arrive in {rng.choice(sorted(DAYS))}"
    else:
        msg = f"reset the password for {rng.choice(['ana', 'ben', 'chen', 'dara'])}@example.com"
    return {"message": msg, "category": cat, "expected": HANDLERS[cat](msg)}


# --- vote world: labelling items, some hard (a fixed decoy label attracts the model) ---
def _build_items() -> dict:
    rng = random.Random(f"{SEED}|items")
    out = {}
    for k in range(N_TASKS):
        true = rng.choice(LABELS)
        hard = rng.random() < float(HARD_SHARE)
        out[f"ticket-{k:03d}"] = {"true": true, "hard": hard, "decoy": rng.choice([x for x in LABELS if x != true])}
    return out


ITEMS = _build_items()

# --- orchestrator world: case files whose number of documents is not known in advance ---
FACTS: dict = {}
CASES: list = []


def _build_cases() -> None:
    rng = random.Random(f"{SEED}|cases")
    for c in range(N_TASKS):
        ids = [f"c{c:03d}d{j}" for j in range(1, rng.randint(2, 7) + 1)]
        for d in ids:
            FACTS[d] = rng.randint(10, 99)
        CASES.append({"ids": ids, "index": ";".join(ids), "expected": ";".join(f"{d}={FACTS[d]}" for d in sorted(ids))})


_build_cases()


# --- refine world: a short status message with four mechanical constraints ---
def check_message(text: str, oid: str) -> list:
    """The executable evaluator: the constraints the text breaks, in a fixed order (empty list = it passes)."""
    out = []
    if oid not in text:
        out.append("oid")
    if len(text.split()) > 12:
        out.append("short")
    if "asap" in text.lower():
        out.append("tone")
    if not text.endswith("Regards, Support"):
        out.append("signoff")
    return out


def compose(oid: str, bad: set) -> str:
    head = f"Order {oid} has shipped" if "oid" not in bad else "Your order has shipped"
    body = " and will arrive soon, ASAP." if "tone" in bad else " and will arrive soon."
    filler = " We are sorry for the long delay and thank you for your patience today." if "short" in bad else ""
    return head + body + filler + (" Regards, Support" if "signoff" not in bad else " Thanks")


class SimModel:
    """A scripted stand-in for a language model. `ask(prompt, sample)` answers `skill: text`; `calls` counts asks."""

    def __init__(self, stream: str, scale: Fraction = Fraction(1)):
        self.stream, self.scale = stream, float(scale)
        self.calls, self.prompts, self.outputs = 0, [], []

    def _p(self, base: float, load: int = 1, extra: float = EXTRA) -> float:
        return min(0.95, self.scale * (base + extra * (load - 1)))

    def _slip(self, name: str, rng: random.Random, load: int) -> bool:
        return rng.random() < self._p(P(name), load)

    def ask(self, prompt: str, sample: int = 0) -> str:
        self.calls += 1
        header, _, payload = prompt.partition(": ")
        skills = header.split("|")
        rng = random.Random(f"{SEED}|{self.stream}|{prompt}|{sample}")
        load = sum(LOAD.get(s, 1) for s in skills)
        for s in skills:
            fn = getattr(self, "_s_" + s, None)
            payload = "ERROR: unknown skill" if fn is None else fn(payload, rng, load)
        self.prompts.append(prompt)
        self.outputs.append(payload)
        return payload

    def _s_extract(self, x, rng, load):
        return _extract_wrong(x, rng) if self._slip("extract", rng, load) else _extract_true(x)

    def _s_total(self, x, rng, load):
        return _total_wrong(x, rng) if self._slip("total", rng, load) else _total_true(x)

    def _s_format(self, x, rng, load):
        return _format_wrong(x, rng) if self._slip("format", rng, load) else _format_true(x)

    def _s_classify(self, x, rng, load):
        cat = category_of(x)
        slip = rng.random() < self._p(P("classify (wrong label)"))
        wrong = rng.choice([c for c in CATEGORIES if c != cat])
        shape = rng.random()
        label = wrong if slip else cat
        if shape < 0.15:
            return label.capitalize() + "."          # right label, untidy surface: not a mistake of meaning
        if shape < 0.17:
            return "unsure"
        return label

    def _handler(self, name, x, rng, load):
        return HANDLERS[name](x, self._slip(name, rng, load))

    def _s_refund(self, x, rng, load):
        return self._handler("refund", x, rng, load)

    def _s_shipping(self, x, rng, load):
        return self._handler("shipping", x, rng, load)

    def _s_reset(self, x, rng, load):
        return self._handler("reset", x, rng, load)

    def _s_answer(self, x, rng, load):
        cat = category_of(x)
        if cat is None:
            return "ERROR: unreadable message"
        return HANDLERS[cat](x, self._slip("answer (one prompt covering all three kinds)", rng, load))

    def _s_label(self, x, rng, load):
        it = ITEMS[x]
        if it["hard"]:
            return it["decoy"] if rng.random() < self._p(P("label, hard item (picks the decoy)")) else it["true"]
        if rng.random() < self._p(P("label, easy item")):
            return rng.choice([lab for lab in LABELS if lab != it["true"]])
        return it["true"]

    def _s_plan(self, x, rng, load):
        ids = x.split(";")
        if len(ids) > 1 and rng.random() < self._p(P("plan drops a document")):
            ids.pop(rng.randrange(len(ids)))
        if rng.random() < self._p(P("plan repeats a document")):
            ids.insert(rng.randrange(len(ids) + 1), rng.choice(ids))
        marker = "- " if rng.random() < 0.3 else ""          # an untidy surface, not an error
        return ("\n\n" if marker else "\n").join(marker + d for d in ids)

    def _s_digest(self, x, rng, load):
        d = x.strip()
        if d not in FACTS:
            return "ERROR: unknown document"
        return f"{d}={FACTS[d] + (1 if self._slip('digest (one document)', rng, load) else 0)}"

    def _s_digest_all(self, x, rng, load):
        ids = x.split(";")
        p = self._p(P("digest (one document)"), len(ids), P("digest_all, extra per added document"))
        lines = []
        for d in ids:
            r = rng.random()
            if r < p / 2:
                continue                                      # a document forgotten
            lines.append(f"{d}={FACTS[d] + (1 if r < p else 0)}")
        return ";".join(sorted(lines))

    def _s_merge(self, x, rng, load):
        lines = [ln for ln in x.split("\n") if ln]
        if len(lines) > 1 and rng.random() < self._p(P("merge drops a line")):
            lines.pop(rng.randrange(len(lines)))
        return ";".join(sorted(lines))

    def _s_draft(self, x, rng, load):
        return compose(x, {c for c in CONSTRAINTS if rng.random() < self._p(P("draft breaks each constraint"))})

    def _s_revise(self, x, rng, load):
        problem, brief, draft = x.split(" :: ", 2)
        bad = set(check_message(draft, brief))
        if rng.random() >= self._p(P("revise leaves the problem")):
            bad.discard(problem)
        if rng.random() < self._p(P("revise breaks another constraint")):
            ok = [c for c in CONSTRAINTS if c not in bad]
            if ok:
                bad.add(rng.choice(ok))
        return compose(brief, bad)


def _pct(x: Fraction) -> str:
    return f"≈{100 * x.numerator / x.denominator:.1f}%"


def show_world() -> None:
    print(f"{'planted tendency (slip rate before scaling)':<52}{'rate':>7}")
    for name, rate in PLANTED.items():
        print(f"  {name:<50}{100 * rate.numerator / rate.denominator:>6.0f}%")
    print(f"  {'hard share of labelling items':<50}{100 * HARD_SHARE.numerator / HARD_SHARE.denominator:>6.0f}%")
    m = SimModel("demo")
    a, b = m.ask("label: ticket-000", 0), m.ask("label: ticket-000", 0)
    print(f"\nthe same prompt and the same sample twice: {a!r} and {b!r}, identical: {a == b}")
    draws = [SimModel("demo").ask("label: ticket-005", s) for s in range(8)]
    print(f"eight different samples of one hard item: {draws}")
    print(f"(the true label of that item is {ITEMS['ticket-005']['true']!r}, its decoy {ITEMS['ticket-005']['decoy']!r})")


if __name__ == "__main__":
    show_world()

## 2. Exercise 1: prompt chaining with a gate and a retry

**Prompt chaining** splits a task into fixed steps; each call takes the previous call's output. The aim is to make each call an
easier job. Between steps your code can put a **gate**: a plain function that checks an intermediate result before the next
step spends a call on it. The task: turn an untidy order line into a sentence, in three steps (`extract`, `total`, `format`),
each with its own gate (`gate_record`, `gate_total`, `gate_sentence`, given above and listed in `STEPS`).

As of 20 October 2026 the article describes prompt chaining as a sequence in which each call processes the previous call's output,
and names the aim as trading latency for accuracy by making each call an easier task. The table below measures that aim inside the
scripted model only; it does not test it on a real one.

**`chain(model, steps, x, retries)`**: `steps` is a list of `(skill, gate)`; `gate` maps the step's output text to
`True` or `False`, or is `None` (accept anything). For each step in order, ask `model.ask(f"{skill}: {x}", sample=attempt)`
with `attempt` = 0, 1, ... up to `retries`, so at most `retries + 1` asks per step. The first output the gate accepts becomes `x`
for the next step exactly as it came back (do not tidy it). If no attempt is accepted, stop at once: return `{"output": None, "failed_at": index}` (the step's index
from 0) and make no further call. After the last step return `{"output": x, "failed_at": None}`.

<details><summary>💡 Hint 1 — what to think about</summary>

A retry is only worth a call if it can come out differently. What does the model's second parameter do? And when a step
fails every attempt, which later steps should still run?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

An outer loop over the steps with their index, an inner loop over attempts that stops at the first accepted output,
and a way to tell, after the inner loop, whether it stopped early or ran out. The accepted output is what the next prompt is built from.

</details>

In [ ]:
def chain(model, steps: list, x: str, retries: int) -> dict:
    """Run `steps` in order, gating and retrying each (see the rules above).

    Example (one step whose gate accepts only upper case, the model answers "abc" then "ABC"):
        >>> chain(model, [("shout", str.isupper)], "abc", 1)    # doctest: +SKIP
        {'output': 'ABC', 'failed_at': None}

    Returns:
        {"output": text or None, "failed_at": index of the step that never passed, or None}
    """
    # YOUR CODE HERE
    raise NotImplementedError


STEPS = [("extract", gate_record), ("total", gate_total), ("format", gate_sentence)]


class _Fake:
    """A model whose answers are a function of (prompt, sample); it records every ask. Used only by the checks."""

    def __init__(self, answer):
        self.answer, self.asked, self.calls = answer, [], 0

    def ask(self, prompt, sample=0):
        self.calls += 1
        self.asked.append((prompt, sample))
        return self.answer(prompt, sample)


def _check_chain() -> None:
    def shout(prompt, sample):
        skill, _, text = prompt.partition(": ")
        return text.upper() if skill == "up" else text + "!"

    fake = _Fake(shout)
    got = chain(fake, [("up", None), ("bang", None)], "abc", 0)
    assert got == {"output": "ABC!", "failed_at": None}, f"got {got}: each step's output is the next step's input, and a clean run has failed_at None"
    assert fake.asked == [("up: abc", 0), ("bang: ABC", 0)], f"asked {fake.asked}: the prompt is f'{{skill}}: {{x}}' with the previous accepted output"
    flaky = _Fake(lambda p, s: "T" if p.startswith("t") else ("good" if s >= 1 else "bad"))
    got = chain(flaky, [("s", lambda t: t == "good"), ("t", None)], "x", 2)
    assert got["output"] == "T" and flaky.asked[:2] == [("s: x", 0), ("s: x", 1)], (
        f"asked {flaky.asked}: a retry must pass the next sample number, or the model repeats its answer")
    assert flaky.calls == 3 and flaky.asked[2] == ("t: good", 0), "the accepted retry feeds the next step, which starts again at sample 0; stop retrying once a gate accepts"
    never = _Fake(lambda p, s: "bad")
    got = chain(never, [("a", None), ("b", lambda t: False), ("c", None)], "x", 1)
    assert got == {"output": None, "failed_at": 1}, f"got {got}: report the index of the step that never passed, from 0"
    assert never.calls == 3, f"{never.calls} calls: step a once, step b twice (retries=1), and step c never"
    padded = _Fake(lambda p, s: "  ABC " if p.startswith("up") else p)
    chain(padded, [("up", None), ("echo", None)], "abc", 0)
    assert padded.asked[1] == ("echo:   ABC ", 0), f"asked {padded.asked[1]}: the accepted output goes on exactly as it came back, so do not strip or tidy it"
    zero = _Fake(lambda p, s: "bad")
    chain(zero, [("a", lambda t: False)], "x", 0)
    assert zero.calls == 1, "retries=0 means one attempt, not none"
    assert chain(_Fake(lambda p, s: "z"), [], "start", 3) == {"output": "start", "failed_at": None}, "no steps: the input comes back unchanged"
    print("exercise 1 looks right")


if __name__ == "__main__":
    _try("exercise 1", _check_chain)

In [ ]:
def _chain_rows(scale: Fraction = Fraction(1)) -> list:
    """Four ways to run the order-line task, each over every task: (label, correct, calls, loud, silent)."""
    rows = []
    ungated = [(s, None) for s, _ in STEPS]
    for label, steps, retries, stream in (("chain, no gates", ungated, 0, "x0"), ("chain, gates, no retry", STEPS, 0, "x1"),
                                          ("chain, gates, one retry", STEPS, RETRIES, "s")):
        correct = calls = loud = silent = 0
        for i in range(N_TASKS):
            t, m = chain_task(i), SimModel(f"chain|{i}|{stream}", scale)
            r = chain(m, steps, t["raw"], retries)
            ok = r["output"] == t["expected"]
            correct, calls = correct + ok, calls + m.calls
            loud += r["failed_at"] is not None
            silent += (r["failed_at"] is None) and not ok
        rows.append((label, correct, Fraction(calls, N_TASKS), loud, silent))
    correct = calls = 0
    for i in range(N_TASKS):
        t, m = chain_task(i), SimModel(f"chain|{i}|b", scale)
        correct, calls = correct + (m.ask("extract|total|format: " + t["raw"]) == t["expected"]), calls + m.calls
    return [("one call with all three jobs", correct, Fraction(calls, N_TASKS), 0, N_TASKS - correct)] + rows


def show_chain() -> None:
    print(f"{'order line -> sentence':<32}{'correct':>9}{'calls/task':>12}{'stopped by':>12}{'wrong, no':>11}")
    print(f"{'':<32}{'of ' + str(N_TASKS):>9}{'':>12}{'a gate':>12}{'gate fired':>11}")
    for label, correct, calls, loud, silent in _chain_rows():
        print(f"{label:<32}{correct:>9}{'≈%.2f' % float(calls):>12}{loud:>12}{silent:>11}")
    print("\na gate turns some wrong answers into loud ones and, with a retry, into right ones; it can only catch what code can check")


if __name__ == "__main__":
    _try("chain table", show_chain, needs=("exercise 1",))

## 3. Exercise 2: routing

**Routing** classifies an input and sends it to a prompt specialised for that kind of input. One prompt that must handle every
kind of customer message carries more jobs than three prompts that each handle one. The price: a classifier call
first, and its mistakes. This model's classifier sometimes answers in an untidy form (`"Refund."`), and sometimes says
`"unsure"`, which names no handler. As of 20 October 2026 the article describes routing as classifying an input and directing it to a
specialised follow-up task, and says it suits inputs with distinct categories that can be classified accurately; the classifier's
mistakes in the table below test that second condition inside the script.

**`route(model, message, labels)`** asks `classify: {message}`, then **normalises** the answer: strip whitespace, lower-case it, and
only then remove any trailing full stops (a full stop is not whitespace: `"refund . "` becomes `"refund "`, which is not a label). If the result is one of `labels`, ask `{label}: {message}` and return
`{"answer": reply, "label": label, "fell_back": False}`. Otherwise ask the generic prompt `answer: {message}` (which covers every
kind) and return `{"answer": reply, "label": None, "fell_back": True}`. Either way that is exactly two asks.

<details><summary>💡 Hint 1 — what to think about</summary>

What does the classifier's text look like before it is clean, and what should happen to a label that is not in `labels`
even though it is a real word, such as a kind of message the caller did not offer?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Normalise first, then a single membership test decides between the specialised prompt and the generic one.
Build both prompts the same way as exercise 1: the skill, a colon and a space, then the message.

</details>

In [ ]:
def route(model, message: str, labels: list) -> dict:
    """Classify `message`, then answer it with the matching specialised prompt or the generic one.

    Example (the classifier says " Refund. "):
        >>> route(model, "please refund ...", ["refund", "reset"])    # doctest: +SKIP
        {'answer': 'REFUND ...', 'label': 'refund', 'fell_back': False}

    Returns:
        {"answer": text, "label": normalised label or None, "fell_back": bool}
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_route() -> None:
    def say(label):
        return _Fake(lambda p, s: label if p.startswith("classify") else f"<{p}>")

    fake = say("  Refund. ")
    got = route(fake, "msg", ["refund", "reset"])
    assert got == {"answer": "<refund: msg>", "label": "refund", "fell_back": False}, f"got {got}: normalise (strip, lower-case, drop trailing full stops) before the test"
    assert fake.asked == [("classify: msg", 0), ("refund: msg", 0)], f"asked {fake.asked}: classify first, then the specialised prompt"
    fake = say("unsure")
    got = route(fake, "msg", ["refund", "reset"])
    assert got == {"answer": "<answer: msg>", "label": None, "fell_back": True} and fake.calls == 2, f"got {got} after {fake.calls} calls: an unknown label falls back to the generic prompt `answer`, still two asks"
    got = route(say("shipping"), "msg", ["refund", "reset"])
    assert got["fell_back"] and got["label"] is None, "a real label that the caller did not offer is also unknown"
    assert route(say("RESET.."), "m", ["reset"])["label"] == "reset", "remove every trailing full stop, not just one"
    assert route(say("re.fund"), "m", ["refund"])["fell_back"], "only TRAILING full stops go; an inner one makes the label unknown"
    assert route(say(".refund"), "m", ["refund"])["fell_back"], "a LEADING full stop is not removed either"
    assert route(say("refund . "), "m", ["refund"])["fell_back"], "strip and lower-case first, then remove only full stops: `refund .` leaves `refund ` with a space, which is no label"
    print("exercise 2 looks right")


if __name__ == "__main__":
    _try("exercise 2", _check_route)

In [ ]:
def show_route() -> None:
    tidy = untidy = fell = misrouted = slipped = routed_ok = single_ok = 0
    for i in range(N_TASKS):
        t = route_task(i)
        m = SimModel(f"route|{i}|s")
        r = route(m, t["message"], CATEGORIES)
        raw = m.outputs[0]
        untidy += raw != raw.strip().lower().rstrip(".")
        fell += r["fell_back"]
        misrouted += (not r["fell_back"]) and r["label"] != t["category"]
        slipped += (not r["fell_back"]) and r["label"] == t["category"] and r["answer"] != t["expected"]
        routed_ok += r["answer"] == t["expected"]
        single_ok += SimModel(f"route|{i}|b").ask("answer: " + t["message"]) == t["expected"]
    print(f"of {N_TASKS} messages: the classifier's text needed tidying {untidy} times, named no handler {fell} times,"
          f" sent {misrouted} to the wrong handler, and the right handler slipped {slipped} times")
    print(f"correct: one prompt for everything {single_ok}, routed {routed_ok}; calls per message: 1 against 2")


if __name__ == "__main__":
    _try("routing table", show_route, needs=("exercise 2",))

## 4. Exercise 3: parallelisation by voting

As of 20 October 2026 the article names two forms of **parallelisation**. *Sectioning* splits a task into independent parts run side by side; the answer does not
change, only the time it takes. *Voting* runs the same task several times and takes the majority. Voting spends calls to
buy confidence, and it helps only where the repeated draws are independent. Here one labelling task has easy items (errors
independent) and hard items (errors correlated, the model prefers the same wrong label each time).

**`majority(answers)`** strips each answer, counts them and returns `(winner, count)`; on a tie the winner is the answer that
appears **first** in the list; an empty list raises `ValueError`.
**`parallel_vote(model, prompt, n)`** asks `n` times, the i-th ask with `sample=i`, and returns
`{"answer": winner, "count": winner's count, "n": n, "agreement": Fraction(count, n)}`; `n < 1` raises `ValueError`.
(The calls are independent, so a real system sends them at once. Here they run one after another and only calls are counted.)

<details><summary>💡 Hint 1 — what to think about</summary>

What would five identical asks give you? And when two answers tie on count, which data structure remembers which came first?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Collect the answers with a loop that passes the loop index as the sample. For the count, a `Counter` over the stripped answers
keeps first-seen order; pick the highest count, taking the earliest on a tie.

</details>

In [ ]:
def majority(answers: list) -> tuple:
    """The most common stripped answer and its count; ties go to the answer seen first.

    Example:
        >>> majority(["b", " a", "a ", "b"])
        ('b', 2)

    Returns:
        (winner, count)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def parallel_vote(model, prompt: str, n: int) -> dict:
    """Ask `n` times with samples 0..n-1 and take the majority (see the rules above).

    Example (a model that answers "x" on samples 0 and 2 and "y" on sample 1):
        >>> parallel_vote(model, "label: t", 3)    # doctest: +SKIP
        {'answer': 'x', 'count': 2, 'n': 3, 'agreement': Fraction(2, 3)}

    Returns:
        {"answer", "count", "n", "agreement"}
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_vote() -> None:
    assert majority(["b", " a", "a ", "b"]) == ("b", 2), f"got {majority(['b', ' a', 'a ', 'b'])}: strip before counting; `b` appeared first, so it wins the 2-2 tie"
    assert majority([" a", "a", "b"]) == ("a", 2), f"got {majority([' a', 'a', 'b'])}: strip each answer before counting, so ' a' and 'a' are one answer"
    assert majority(["x"]) == ("x", 1) and majority(["q", "r", "s"]) == ("q", 1), "a three-way tie goes to the first answer seen"
    assert majority(["r", "q", "q"]) == ("q", 2), "a clear majority beats the first-seen rule"
    try:
        majority([])
    except ValueError:
        pass
    else:
        raise AssertionError("an empty list has no winner: raise ValueError")
    fake = _Fake(lambda p, s: ["x", "y", "x"][s])
    got = parallel_vote(fake, "label: t", 3)
    assert got == {"answer": "x", "count": 2, "n": 3, "agreement": Fraction(2, 3)}, f"got {got}"
    assert [s for _, s in fake.asked] == [0, 1, 2], f"samples used {[s for _, s in fake.asked]}: the i-th ask needs sample=i, or all the draws are the same draw"
    assert isinstance(got["agreement"], Fraction), "agreement is an exact Fraction"
    one = parallel_vote(_Fake(lambda p, s: "z"), "p", 1)
    assert one["answer"] == "z" and one["agreement"] == 1, "n=1 is a plain call"
    try:
        parallel_vote(_Fake(lambda p, s: "z"), "p", 0)
    except ValueError:
        pass
    else:
        raise AssertionError("n=0 must raise ValueError")
    print("exercise 3 looks right")


if __name__ == "__main__":
    _try("exercise 3", _check_vote)

In [ ]:
def show_vote() -> None:
    keys = [f"ticket-{i:03d}" for i in range(N_TASKS)]
    hard = [i for i, k in enumerate(keys) if ITEMS[k]["hard"]]
    easy = [i for i in range(N_TASKS) if i not in hard]
    # One stream per task for every row: a row with more draws is the row with fewer draws plus extra draws (the same sample numbers
    # give the same answers), so the rows differ only by the draws added, and one draw is exactly the single call of section 7.
    votes = {n: [parallel_vote(SimModel(f"vote|{i}|s"), "label: " + keys[i], n) for i in range(N_TASKS)] for n in (1, 3, 5, 9, 15)}
    right = {n: [r["answer"] == ITEMS[k]["true"] for r, k in zip(votes[n], keys)] for n in votes}
    print(f"{'draws':>6}{'correct':>9}{'easy items':>12}{'hard items':>12}{'calls/item':>12}")
    print(f"{'':>6}{'of ' + str(N_TASKS):>9}{'of ' + str(len(easy)):>12}{'of ' + str(len(hard)):>12}")
    for n in votes:
        print(f"{n:>6}{sum(right[n]):>9}{sum(right[n][i] for i in easy):>12}{sum(right[n][i] for i in hard):>12}{n:>12}")
    mean = lambda xs: sum(xs, Fraction(0)) / len(xs) if xs else Fraction(0)
    print(f"\nmean agreement of the {VOTES}-draw vote, by kind of item and by whether the vote was right:")
    print(f"{'':<8}{'right':>8}{'items':>7}{'wrong':>9}{'items':>7}")
    for name, group in (("easy", easy), ("hard", hard), ("all", range(N_TASKS))):
        ok = [votes[VOTES][i]["agreement"] for i in group if right[VOTES][i]]
        bad = [votes[VOTES][i]["agreement"] for i in group if not right[VOTES][i]]
        print(f"{name:<8}{_pct(mean(ok)):>8}{len(ok):>7}{_pct(mean(bad)) if bad else '-':>9}{len(bad):>7}")
    gap = lambda group: (mean([votes[VOTES][i]["agreement"] for i in group if right[VOTES][i]])
                         - mean([votes[VOTES][i]["agreement"] for i in group if not right[VOTES][i]]))
    print(f"gap in mean agreement, right minus wrong: ≈{100 * float(gap(easy)):+.1f} points on easy items, ≈{100 * float(gap(hard)):+.1f} points on hard items")
    ns = list(votes)
    dips = [f"{a} to {b} draws: total {sum(right[b]) - sum(right[a]):+d} (easy {sum(right[b][i] for i in easy) - sum(right[a][i] for i in easy):+d}, "
            f"hard {sum(right[b][i] for i in hard) - sum(right[a][i] for i in hard):+d})" for a, b in zip(ns, ns[1:]) if sum(right[b]) < sum(right[a])]
    print("rows where more draws scored fewer correct in total: " + ("; ".join(dips) if dips else "none") + ".")
    one, many = sum(right[1][i] for i in hard), sum(right[VOTES][i] for i in hard)
    easy_one, easy_many = sum(right[1][i] for i in easy), sum(right[VOTES][i] for i in easy)
    print(f"\non the easy items {VOTES} draws got {easy_many} right against {easy_one} for one draw; on the hard items {many} against {one}.")
    print("By the planted rules the draws on a hard item mostly pick the same decoy, so a vote has no independent draws to average there;"
          " agreement then measures how consistent the draws are, not whether they are right.")
    print("The rows share their draws (one stream per item), so a row differs from the one before it only through the draws added.")


if __name__ == "__main__":
    _try("vote curve", show_vote, needs=("exercise 3",))

## 5. Exercise 4: orchestrator-workers

Chaining and routing fix their steps in advance. When you cannot know the sub-tasks until you see the input, an
**orchestrator** plans them, **workers** each do one, and a **synthesiser** combines the results. The task: a case file of an
unknown number of documents; the answer lists every document's key fact. One call over all the documents forgets more of
them the more there are. The orchestrator's plan is itself model output, so it can drop a document, repeat one or come back in an untidy list.
As of 20 October 2026 the article describes this workflow as a central model that breaks tasks down dynamically, delegates them to worker
models and synthesises their results, and says it differs from parallelisation in that the sub-tasks are not predefined but determined from
the input; the case files here have an unknown number of documents for that reason.

**`parse_plan(text)`** returns the planned document ids, one per line: strip each line, remove one leading `- ` or `* ` marker
and strip again, skip empty lines, and keep each id only once, in first-seen order. Ids are case-sensitive and kept as written.
**`orchestrate(model, index)`** asks `plan: {index}` and parses it. With no sub-task it returns `{"answer": "", "plan": []}`
after that one call. Otherwise it asks `digest: {sub}` once per sub-task in plan order, then once `merge: ` followed by the worker
replies joined with `"\n"`, and returns `{"answer": the merge reply, "plan": the sub-tasks}`. Every ask here uses the default `sample` of 0.

<details><summary>💡 Hint 1 — what to think about</summary>

The plan arrives as text, and a planner can add bullets, blank lines and repeats. What would a repeated id cost, if it
became two worker calls and two lines in the merge? And what should happen when the plan is empty?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Build the id list line by line, checking whether you have it already. Then one comprehension for the workers, in order,
and a single merge ask at the end. Return early before the workers when the list is empty.

</details>

In [ ]:
def parse_plan(text: str) -> list:
    """The ids in a planner's reply: one per line, markers removed, each once, in first-seen order.

    Example:
        >>> parse_plan("- d1\\n\\n* d2\\n d1 \\nd3")
        ['d1', 'd2', 'd3']

    Returns:
        A list of ids.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def orchestrate(model, index: str) -> dict:
    """Plan, run one worker per sub-task, then synthesise (see the rules above).

    Example (the planner answers "d1\\nd2"):
        >>> orchestrate(model, "d1;d2")    # doctest: +SKIP
        {'answer': 'd1=5;d2=7', 'plan': ['d1', 'd2']}

    Returns:
        {"answer": text, "plan": list of sub-tasks}
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_orchestrate() -> None:
    got = parse_plan("- d3\n\n* d1\n d3 \nd2")
    assert got == ["d3", "d1", "d2"], f"got {got}: strip, drop one marker, skip blanks, keep each id once in first-seen order"
    assert parse_plan("") == [] and parse_plan("\n  \n") == [], "blank lines are not sub-tasks"
    assert parse_plan("-  d1") == ["d1"], "strip again after removing the marker"
    assert parse_plan("-d1") == ["-d1"], "a marker is `- ` or `* ` with the space; `-d1` is an id"
    assert parse_plan("D1\nd1\nD1") == ["D1", "d1"], "ids are case-sensitive: keep them as written, so `D1` and `d1` are two ids"

    def planner(prompt, sample):
        skill, _, text = prompt.partition(": ")
        return {"plan": "- b\n\n- a\n- b", "digest": text + "=1", "merge": "M"}[skill]

    fake = _Fake(planner)
    got = orchestrate(fake, "a;b")
    assert got == {"answer": "M", "plan": ["b", "a"]}, f"got {got}"
    assert [p for p, _ in fake.asked] == ["plan: a;b", "digest: b", "digest: a", "merge: b=1\na=1"], (
        f"asked {[p for p, _ in fake.asked]}: plan once, one digest per DISTINCT sub-task in plan order, one merge of the replies joined by a newline")
    assert all(sample == 0 for _, sample in fake.asked), f"samples used {[sample for _, sample in fake.asked]}: every ask in this exercise uses the default sample 0"
    empty = _Fake(lambda p, s: "")
    got = orchestrate(empty, "a;b")
    assert got == {"answer": "", "plan": []} and empty.calls == 1, f"got {got} after {empty.calls} calls: an empty plan stops after the planning call"
    print("exercise 4 looks right")


if __name__ == "__main__":
    _try("exercise 4", _check_orchestrate)

In [ ]:
def show_orchestrate() -> None:
    dropped = repeated = bulleted = 0
    buckets = {"2-3 documents": range(2, 4), "4-5 documents": range(4, 6), "6-7 documents": range(6, 8)}
    tally = {b: [0, 0, 0] for b in buckets}                         # cases, orchestrated correct, single call correct
    for i, c in enumerate(CASES):
        m = SimModel(f"orchestrate|{i}|s")
        r = orchestrate(m, c["index"])
        raw = m.outputs[0]
        dropped += set(r["plan"]) != set(c["ids"])
        repeated += len(re.findall(r"c\d+d\d", raw)) != len(set(re.findall(r"c\d+d\d", raw)))
        bulleted += raw.startswith("- ")
        row = tally[next(b for b, rng in buckets.items() if len(c["ids"]) in rng)]
        row[0] += 1
        row[1] += r["answer"] == c["expected"]
        row[2] += SimModel(f"orchestrate|{i}|b").ask("digest_all: " + c["index"]) == c["expected"]
    print(f"of {N_TASKS} case files: the plan dropped a document {dropped} times, repeated one {repeated} times, and came back as a bulleted list {bulleted} times")
    print(f"\n{'size of case file':<18}{'cases':>7}{'one call':>10}{'orchestrated':>14}")
    for b, (n, orch, single) in tally.items():
        print(f"{b:<18}{n:>7}{single:>10}{orch:>14}   (correct)")
    print("\nthe single call gets worse as the file grows; the orchestrator pays a fixed risk (a dropped document, a dropped line) and calls per document")


if __name__ == "__main__":
    _try("orchestrator table", show_orchestrate, needs=("exercise 4",))

## 6. Exercise 5: evaluate and refine

**Evaluate-and-refine** has one call produce a draft, an evaluator check it, and a reviser repair it, in a loop. It pays when you have
clear criteria and when feedback demonstrably improves the draft. The criteria here are four mechanical rules (the order id is
present, at most twelve words, no "ASAP", the sign-off ends the text) checked by `check_message`, **executable code**,
so the evaluator cannot be talked round. The reviser sometimes leaves the problem and sometimes breaks another rule.
As of 20 October 2026 the article describes the workflow as one call generating a response while another gives evaluation and feedback
in a loop, and says it is effective when there are clear evaluation criteria; the noisy judge below shows what a weaker evaluator costs.

**`refine(model, evaluate, brief, max_rounds)`** asks `draft: {brief}`, then calls `evaluate(draft)`, which returns the list
of problems (empty means it passes). While there is a problem and fewer than `max_rounds` revisions have been made,
ask `revise: {first problem} :: {brief} :: {draft}` for a new draft and evaluate it. Return
`{"answer": last draft, "history": every evaluate result in order (the first draft's included), "accepted": the last result is empty, "revisions": how many revise asks}`.
The last draft is always evaluated, so `accepted` describes the draft you return.

<details><summary>💡 Hint 1 — what to think about</summary>

How many times is `evaluate` called when `max_rounds` is zero? When the first draft passes? When the loop ends because the cap is
reached, has the draft you are returning been evaluated?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Keep the history list as the loop's state: its last entry says whether to go on, and its length minus one is the number
of revisions so far. A `while` with both conditions, and one evaluate call per new draft, no more.

</details>

In [ ]:
def refine(model, evaluate: Callable, brief: str, max_rounds: int) -> dict:
    """Draft, evaluate, and revise the first reported problem, at most `max_rounds` times (see the rules above).

    Example (the first draft fails with ["oid"], the revision passes):
        >>> refine(model, evaluate, "A1", 3)    # doctest: +SKIP
        {'answer': '...', 'history': [['oid'], []], 'accepted': True, 'revisions': 1}

    Returns:
        {"answer", "history", "accepted", "revisions"}
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_refine() -> None:
    def say(prompt, sample):
        skill, _, text = prompt.partition(": ")
        if skill == "draft":
            return "v0"
        return "v" + str(int(text.split(" :: ")[2][1:]) + 1)

    seen = []

    def evaluate(d):
        seen.append(d)
        return [] if d == "v2" else ["p" + d[1:], "q"]

    fake = _Fake(say)
    got = refine(fake, evaluate, "B", 5)
    assert got == {"answer": "v2", "history": [["p0", "q"], ["p1", "q"], []], "accepted": True, "revisions": 2}, f"got {got}"
    assert fake.asked[1][0] == "revise: p0 :: B :: v0", f"revise prompt was {fake.asked[1][0]!r}: the FIRST problem, then the brief, then the current draft, joined by ' :: '"
    assert seen == ["v0", "v1", "v2"], f"evaluate saw {seen}: once per draft, in order, and not again after it passes"
    seen.clear()
    fake = _Fake(say)
    got = refine(fake, lambda d: seen.append(d) or ["p"], "B", 2)
    assert got["revisions"] == 2 and not got["accepted"] and got["answer"] == "v2" and seen == ["v0", "v1", "v2"] and fake.calls == 3, (
        f"got {got}, evaluate saw {seen}: with a cap of 2 there are 2 revisions, and the last draft is still evaluated")
    fake = _Fake(say)
    got = refine(fake, lambda d: ["p"], "B", 0)
    assert got == {"answer": "v0", "history": [["p"]], "accepted": False, "revisions": 0} and fake.calls == 1, f"got {got}: a cap of zero revises nothing but still evaluates the draft"
    fake = _Fake(say)
    got = refine(fake, lambda d: [], "B", 3)
    assert got["revisions"] == 0 and got["accepted"] and fake.calls == 1, "a first draft that passes is returned as it is"
    print("exercise 5 looks right")


if __name__ == "__main__":
    _try("exercise 5", _check_refine)

In [ ]:
class NoisyJudge:
    """An evaluator that stands in for a model reviewing its own kind of output: when the draft has a problem it
    still says "no problems" with probability `accept`. (Seeded from the draft, so the same draft gets the same verdict.)"""

    def __init__(self, stream: str, evaluate: Callable, accept: Fraction = Fraction(35, 100)):
        self.stream, self.evaluate, self.accept = stream, evaluate, float(accept)

    def __call__(self, draft: str) -> list:
        problems = self.evaluate(draft)
        if problems and random.Random(f"{SEED}|judge|{self.stream}|{draft}").random() < self.accept:
            return []
        return problems


def show_refine() -> None:
    def run(rounds, judged):
        ok = calls = accepted_bad = 0
        for i in range(N_TASKS):
            oid = f"A{1000 + i}"
            m = SimModel(f"refine|{i}|s")   # one stream per task for every cap: a larger cap only continues the same trajectory
            ev = (lambda d, oid=oid: check_message(d, oid))
            r = refine(m, NoisyJudge(f"{i}", ev) if judged else ev, oid, rounds)
            good = check_message(r["answer"], oid) == []
            ok, calls, accepted_bad = ok + good, calls + m.calls, accepted_bad + (r["accepted"] and not good)
        return ok, Fraction(calls, N_TASKS), accepted_bad

    print(f"{'revision cap':>13}{'correct':>9}{'calls/task':>12}      (executable evaluator, of {N_TASKS}; cap 0 is the single call of section 7)")
    for rounds in (0, 1, 2, 3, 5):
        ok, calls, _ = run(rounds, False)
        print(f"{rounds:>13}{ok:>9}{'≈%.2f' % float(calls):>12}")
    ok, calls, bad = run(3, True)
    print(f"\nthe same loop with a noisy judge in place of check_message, cap 3: correct {ok}, calls/task ≈{float(calls):.2f},"
          f" accepted although failing {bad}")
    print("the loop is only as good as its evaluator: a judge that waves bad drafts through caps the result")


if __name__ == "__main__":
    _try("refine table", show_refine, needs=("exercise 5",))

## 7. Does the structure pay? The five patterns against one call

Every pattern ran the same recipe: the same tasks, an executable check, and a single call (or the nearest single-call
version) as the baseline. That is the article's advice, as of 20 October 2026, to start simple and add structure only when simpler solutions fall short. For each pattern the next cell prints the correct count, calls per task, and the **paired** result: tasks the pattern got
right and the baseline wrong (gains), and the reverse (losses). A sign test (`sign_test`, given) turns gains and losses into **p**, the chance of a
split at least this lopsided if both kinds of change were equally likely. It describes this sample of tasks only; with other tasks the counts would move.
Then the failure modes each pattern adds, counted from what the episode did, not from the answer alone.

In [ ]:
def sign_test(gains: int, losses: int) -> Fraction:
    """Exact two-sided sign-test p-value for `gains` against `losses` (ties excluded): 2 * P(X <= min) for X ~ Binomial(n, 1/2), at most 1."""
    n = gains + losses
    if n == 0:
        return Fraction(1)
    return min(Fraction(1), Fraction(2 * sum(comb(n, k) for k in range(min(gains, losses) + 1)), 2 ** n))


KINDS = ["chain", "route", "vote", "orchestrate", "refine"]
KIND_LABEL = {"chain": "chaining", "route": "routing", "vote": "voting", "orchestrate": "orchestrator", "refine": "evaluate-refine"}


def episode(kind: str, i: int, structured: bool, scale: Fraction) -> dict:
    """One task with the pattern (structured) or with its single-call baseline. Returns ok, calls, failure modes, repairs.

    For voting and for evaluate-and-refine the baseline shares the pattern's stream: the single call IS the pattern's first draw
    (sample 0, the first draft), so the pattern's extra work is judged against exactly what it started from, in every table."""
    s = f"{kind}|{i}|{'s' if structured or kind in ('vote', 'refine') else 'b'}"
    m, modes, repairs = SimModel(s, scale), [], []
    if kind == "chain":
        t = chain_task(i)
        if structured:
            r = chain(m, STEPS, t["raw"], RETRIES)
            ok = r["output"] == t["expected"]
            if r["failed_at"] is not None:
                modes.append("stopped by a gate (loud)")
            elif not ok:
                modes.append("wrong, no gate fired (silent)")
            elif m.calls > len(STEPS):
                repairs.append("a gate and a retry turned a slip into a right answer")
        else:
            ok = m.ask("extract|total|format: " + t["raw"]) == t["expected"]
    elif kind == "route":
        t = route_task(i)
        if structured:
            r = route(m, t["message"], CATEGORIES)
            ok = r["answer"] == t["expected"]
            if r["fell_back"]:
                modes.append("unknown label, generic fallback")
            elif r["label"] != t["category"]:
                modes.append("misrouted")
            elif not ok:
                modes.append("right route, handler slipped")
        else:
            ok = m.ask("answer: " + t["message"]) == t["expected"]
    elif kind == "vote":
        key = f"ticket-{i:03d}"
        if structured:
            r = parallel_vote(m, "label: " + key, VOTES)
            ok = r["answer"] == ITEMS[key]["true"]
            if ok and m.outputs[0].strip() != ITEMS[key]["true"]:
                repairs.append("the first draw was wrong and the vote outvoted it")
            if not ok and r["count"] == r["n"]:
                modes.append("unanimous and wrong")
            elif not ok and 2 * r["count"] > r["n"]:
                modes.append("wrong by a majority")
            elif not ok:
                modes.append("wrong by a plurality")
        else:
            ok = m.ask("label: " + key) == ITEMS[key]["true"]
    elif kind == "orchestrate":
        c = CASES[i]
        if structured:
            r = orchestrate(m, c["index"])
            ok = r["answer"] == c["expected"]
            if set(r["plan"]) != set(c["ids"]):
                modes.append("plan dropped a document")
            elif not ok:
                modes.append("worker or merge slipped")
        else:
            ok = m.ask("digest_all: " + c["index"]) == c["expected"]
    else:
        oid = f"A{1000 + i}"
        ev = (lambda d: check_message(d, oid))
        if structured:
            r = refine(m, ev, oid, ROUNDS)
            ok = check_message(r["answer"], oid) == []
            if ok and r["revisions"] > 0:
                repairs.append("a revision fixed a failing draft")
            h = r["history"]
            if any(set(b) - set(a) for a, b in zip(h, h[1:])):
                modes.append("a revision broke a new rule")
            if not r["accepted"]:
                modes.append("not converged within the cap")
        else:
            ok = check_message(m.ask("draft: " + oid), oid) == []
    return {"ok": ok, "calls": m.calls, "modes": modes, "repairs": repairs}


_RUNS: dict = {}


def run_all(scale: Fraction) -> dict:
    """Every pattern and its baseline over every task at one slip scale; cached, because both tables read it."""
    if scale not in _RUNS:
        _RUNS[scale] = {k: ([episode(k, i, False, scale) for i in range(N_TASKS)],
                            [episode(k, i, True, scale) for i in range(N_TASKS)]) for k in KINDS}
    return _RUNS[scale]


def show_scoreboard() -> None:
    runs = run_all(Fraction(1))
    print(f"{'pattern':<17}{'correct (of ' + str(N_TASKS) + ')':>20}{'calls/task':>16}{'paired vs the single call':>30}")
    print(f"{'':<17}{'single':>10}{'pattern':>10}{'single':>8}{'pattern':>8}{'gains':>8}{'losses':>8}{'p':>7}")
    for k in KINDS:
        base, pat = runs[k]
        cb, cp = sum(e["ok"] for e in base), sum(e["ok"] for e in pat)
        gains = sum(1 for b, p in zip(base, pat) if p["ok"] and not b["ok"])
        losses = sum(1 for b, p in zip(base, pat) if b["ok"] and not p["ok"])
        p_val = sign_test(gains, losses)
        print(f"{KIND_LABEL[k]:<17}{cb:>10}{cp:>10}{'≈%.2f' % (sum(e['calls'] for e in base) / N_TASKS):>8}"
              f"{'≈%.2f' % (sum(e['calls'] for e in pat) / N_TASKS):>8}{gains:>8}{losses:>8}{('<0.001' if p_val < Fraction(1, 1000) else '≈%.3f' % float(p_val)):>7}")
    print("\nwhat went wrong inside each pattern (episodes out of", N_TASKS, "that showed it; some still ended right, e.g. a fall-back that answered correctly).")
    print("The single call has no gate, vote, plan or loop to look inside, so only its wrong answers are counted (the 'single' column above).")
    for k in KINDS:
        modes = Counter(m for e in runs[k][1] for m in e["modes"])
        print(f"  {KIND_LABEL[k]:<17}" + ("; ".join(f"{m} {n}" for m, n in sorted(modes.items())) or "none"))
    print("\nwhat the pattern repaired (episodes that ended right because of the extra work):")
    for k in KINDS:
        reps = Counter(m for e in runs[k][1] for m in e["repairs"])
        print(f"  {KIND_LABEL[k]:<17}" + ("; ".join(f"{m} {n}" for m, n in sorted(reps.items())) or "none counted for this pattern"))
    print("\nthese are results for the scripted model and its printed tendencies, not rates for any real model")


if __name__ == "__main__":
    _try("scoreboard", show_scoreboard,
         needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 8. When does the added structure pay for itself?

As of 20 October 2026 the article says agentic systems often trade latency and cost for better task performance. This section counts calls as
a stand-in for that trade (it does not measure latency or money). A pattern that adds correct answers but costs several calls each may or may not be worth it. The measure to read is
**extra calls per additional correct answer**: the pattern's extra calls over all tasks divided by its extra correct answers. Run it
at several values of `scale`, which multiplies every slip rate. At zero, the model never slips and the baseline is already perfect.

In [ ]:
def show_pays() -> None:
    scales = [Fraction(0), Fraction(1, 2), Fraction(1), Fraction(3, 2)]
    print("extra correct answers, pattern minus single call (of", N_TASKS, "tasks)")
    print(f"{'pattern':<17}" + "".join(f"{'scale ' + str(s):>11}" for s in scales))
    for k in KINDS:
        print(f"{KIND_LABEL[k]:<17}" + "".join(f"{sum(e['ok'] for e in run_all(s)[k][1]) - sum(e['ok'] for e in run_all(s)[k][0]):>+11}" for s in scales))
    print("\nextra calls per additional correct answer (lower is cheaper; 'no gain' when the pattern added nothing)")
    print(f"{'pattern':<17}" + "".join(f"{'scale ' + str(s):>11}" for s in scales))
    for k in KINDS:
        cells = []
        for s in scales:
            base, pat = run_all(s)[k]
            gain = sum(e["ok"] for e in pat) - sum(e["ok"] for e in base)
            extra = sum(e["calls"] for e in pat) - sum(e["calls"] for e in base)
            cells.append("no gain" if gain <= 0 else "≈%.1f" % (extra / gain))
        print(f"{KIND_LABEL[k]:<17}" + "".join(f"{c:>11}" for c in cells))
    print("\nextra calls per task, pattern minus single call (whether or not the answers were right)")
    print(f"{'pattern':<17}" + "".join(f"{'scale ' + str(s):>11}" for s in scales))
    extra0 = {}
    for k in KINDS:
        cells = []
        for s in scales:
            base, pat = run_all(s)[k]
            diff = Fraction(sum(e["calls"] for e in pat) - sum(e["calls"] for e in base), N_TASKS)
            cells.append("≈%.2f" % float(diff))
            if s == 0:
                extra0[k] = diff
        print(f"{KIND_LABEL[k]:<17}" + "".join(f"{c:>11}" for c in cells))
    gained0 = sum(sum(e["ok"] for e in run_all(Fraction(0))[k][1]) - sum(e["ok"] for e in run_all(Fraction(0))[k][0]) for k in KINDS)
    spend = [KIND_LABEL[k] for k in KINDS if extra0[k] > 0]
    free = [KIND_LABEL[k] for k in KINDS if extra0[k] == 0]
    print(f"\nat scale 0 the five patterns add {gained0} correct answers between them; {len(spend)} of the {len(KINDS)} still spend extra calls ({', '.join(spend)})"
          + (f", and {', '.join(free)} spends none, since a retry or a revision is only asked for when a check fails and none fails" if free else "")
          + ". Structure answers a failure rate; it does not create accuracy.")


if __name__ == "__main__":
    _try("pay-off table", show_pays, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 9. What this measurement tells you, and what it does not

* **It tells you about mechanics.** Where each pattern gets its gain (fewer jobs per call, a gate that catches what code can check,
  a specialised prompt, independent draws, a planned decomposition, a loop with a real evaluator) and where it leaks (a gate that cannot
  see a wrong value, a misroute, correlated votes, a dropped sub-task, a revision that breaks a rule, a judge that waves bad drafts through).
* **It neither confirms nor refutes the vendor's guidance.** Every statement attributed above to Anthropic's article is dated 20 October 2026
  and describes the article, not this script. The script has planted rules, so a result here can illustrate why a pattern should help or fail;
  it cannot show that the article is right or wrong about real models.
* **It does not give a rate.** Every rate, the way load costs accuracy, the decoy items and the share of hard ones are the author's
  assumptions. A real model might slip more or less on any of them. The tasks and settings were fixed before any run and no seed was searched,
  which makes the numbers repeatable, not representative.
* **Calls are not time or money.** Parallel calls overlap in time but each costs; a chain's calls add up in time. Latency, tokens and
  price are not measured. The article's remark on latency and cost (section 8) is a statement about real systems, which this script does not measure.
* **Sectioning and agents are named, not built.** Sectioning changes speed, not the answer. A free-running agent
  (T07-L01) chooses its own path; compare it to the best workflow only after the workflow baseline above exists.
* **Real models need your own run.** To test one, write a class whose `ask(prompt, sample)` calls the model and returns its text, and run the same functions.

## 10. Common mistakes

* **Retrying with the same sample.** A retry that cannot differ is a wasted call. Exercise 1's check catches it.
* **Trusting agreement.** Five matching votes show that the draws are consistent. Where the draws share a bias, as on the hard items, they
  can be consistent about the wrong label; section 4 prints how a vote fares on each kind of item.
* **Forgetting the fall-back.** A label the router does not know is data, not an exception.
* **Evaluating the wrong draft.** The draft you return must be the one that was evaluated last.
* **Judging a pattern by accuracy alone.** Read the calls, the losses and the failure modes beside it.

In [ ]:
# Facts about the setup you just ran, printed by the code that runs here, for the self-check below.
if __name__ == "__main__":
    print(f"{len(KINDS)} patterns, {N_TASKS} tasks each; settings: {VOTES} votes, {RETRIES} retry per chain step, at most {ROUNDS} revisions")
    print("each task and each model object is seeded from the pattern, the task number and whether it is the pattern or the baseline")

## 11. Self-check

1. Suppose the gated chain scores above the one-call baseline in section 2's table. Which conclusion does the lesson's design support?
   - (a) Splitting a task into steps never costs anything, so a chain beats one call on any task and any model.
   - (b) The gates made the model smarter, because a model that knows it will be checked answers more carefully.
   - (c) In this script a call slips more as it carries more jobs and a gate stops some slips early; whether a real model's slips grow with load needs its own test.
   - (d) The retry did all of the work, so the gates could be removed without changing what the table shows.

2. On a hard item the script picks the decoy with the planted rate on each draw, and the draws are otherwise independent. Roughly how often is the majority of
   five draws the decoy, and what follows for a high agreement score on such an item?
   - (a) About 4 times in 100, so a high agreement is a reliable sign that the answer is right.
   - (b) About 1 time in 3, so a high agreement usually means the draws found the true label.
   - (c) About 99 times in 100, so every hard item is lost and a vote should never be used.
   - (d) About 2 times in 3, so a vote does worse there than one draw, and a high agreement can be agreement on the decoy.

3. At a slip scale of zero the single call is already right on every task. Which statement holds for all five patterns?
   - (a) Each adds correct answers, because extra structure is a form of correctness.
   - (b) None can add a correct answer; whether one still spends extra calls depends on whether it ever needs a second call.
   - (c) None can run, because a slip probability of zero is not allowed.
   - (d) Each spends extra calls, because a gate, a vote, a plan or a loop always makes at least one more ask.

4. Why does evaluate-and-refine use `check_message` (code) rather than another model call as its evaluator?
   - (a) The loop can only improve a draft as far as its evaluator tells good from bad; an executable check cannot be talked round, while a judge that accepts failing drafts lets them through.
   - (b) Code is always cheaper than a model call, so the loop with a code evaluator costs fewer calls at every cap.
   - (c) A model call cannot read a draft that another model wrote, so only code can act as the evaluator.
   - (d) Code can see the whole draft at once, while a model call is shown only the first problem in it.

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

## 12. Sources, with the date each was read
Every dated statement above is a `claims.yaml` entry with its source, a verbatim quote and the day it was read (20 October 2026).
Vendor guidance changes often: read the page again before you rely on it.
* Anthropic, "Building effective agents" (19 December 2024): workflows against agents; the five workflows (prompt chaining, routing,
  parallelization, orchestrator-workers, evaluator-optimizer); start simple and add structure only when simpler solutions fall short.

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, shown, stubs, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "") and waiting not in stubs
                else "")
        print(f"  {_MARKS[status]}  {label} · {shown}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, shown, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero; in a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.